This file is for testing stochastic depth as a regularization technique

In [1]:
import pandas as pd
import regex as re
import torch

In [2]:
df = pd.read_csv(r'D:\Traffic\labels_processed.csv')

In [3]:
def label_function(dpath):
    class_name = re.findall(r'(\d+)_.*\.png$', dpath.name)
    class_id = int(class_name[0])
    return class_id

In [4]:
from pathlib import Path

In [5]:
path = Path(r'D:\Traffic\traffic_Data_processed\DATA')

In [6]:
lr_head = 0.0017378008365631102
lr_whole_model = 1.9054607491852948e-06

In [7]:
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [8]:
from torch.utils.data import Dataset
from PIL import Image

In [9]:
class dset(Dataset):
    def __init__(self, image_paths, transform = None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert("RGB")
        label = label_function(Path(image_path))
        if self.transform:
            image = self.transform(image)
        return image, label

In [10]:
image_paths = list(path.rglob("*.png"))

In [11]:
from torchvision import transforms
from torch.utils.data import random_split
from torch.utils.data import DataLoader

In [12]:
import torchvision
num_classes = 55
device = torch.device("cuda")

In [13]:
import kornia.augmentation as K
import torch.nn as nn

In [14]:
from torchvision.ops import stochastic_depth

class StochasticBasicBlock(nn.Module):
    def __init__(self, block, drop_prob):
        super().__init__()

        self.conv1 = block.conv1
        self.bn1 = block.bn1
        self.relu = block.relu
        self.conv2 = block.conv2
        self.bn2 = block.bn2
        self.downsample = block.downsample
        self.drop_prob = drop_prob

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = stochastic_depth(
            out,
            p=self.drop_prob,
            mode="row",
            training=self.training
        )

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out

def add_stochastic_depth(model, max_drop_prob=0.2):

    blocks = []

    for layer in [model.layer1, model.layer2, model.layer3, model.layer4]:
        for block in layer:
            blocks.append(block)

    total_blocks = len(blocks)
    block_idx = 0

    for layer in [model.layer1, model.layer2, model.layer3, model.layer4]:
        for i in range(len(layer)):

            drop_prob = max_drop_prob * block_idx / (total_blocks - 1)

            layer[i] = StochasticBasicBlock(
                layer[i],
                drop_prob
            )

            block_idx += 1

    return model

In [15]:
transform_1 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

dataset_1 = dset(image_paths=image_paths, transform=transform_1)

ts = int(0.75*len(dataset_1))
vs = len(dataset_1) - ts
generator = torch.Generator().manual_seed(42)
train_dataset, valid_dataset = random_split(dataset_1, [ts, vs], generator)

train_loader_1 = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

valid_loader_1 = DataLoader(
    valid_dataset,
    batch_size=16,
    shuffle=False
)

model_1 = torchvision.models.resnet34(weights="DEFAULT")

model_1 = add_stochastic_depth(model_1, max_drop_prob=0.2)

model_1.fc = nn.Linear(
    model_1.fc.in_features,
    num_classes
)

model_1 = model_1.to(device)

for param in model_1.parameters():
    param.requires_grad = True

train_aug_1 = K.AugmentationSequential(
    K.ColorJiggle(
        contrast=0.2,
        p=0.5
    ),
    K.RandomPlanckianJitter(
        mode="CIED",
        p=0.5
    )
).to(device)

targeted_aug_1 = K.AugmentationSequential(
    K.RandomRotation(
        degrees=10,
        p=0.5
    ),
    K.RandomAffine(
        degrees=0,
        scale=(0.9, 1.1),
        p=0.5
    ),
    K.RandomPerspective(
        distortion_scale=0.2,
        p=0.5
    ),
    K.ColorJiggle(
        brightness=0.2,
        contrast=0.2,
        p=0.5
    )
).to(device)

optimizer_1 = torch.optim.RMSprop(
    model_1.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_1 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_1,
    max_lr=lr_head,
    epochs=20,
    steps_per_epoch=len(train_loader_1)
)

criterion = nn.CrossEntropyLoss()

for epoch in range(20):
    model_1.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_1:

        images = images.to(device)
        labels = labels.to(device)

        images = train_aug_1(images)

        mask = (labels == 35) | (labels == 36)

        if mask.any():
            images[mask] = targeted_aug_1(images[mask])

        optimizer_1.zero_grad()

        outputs = model_1(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_1.step()

        scheduler_1.step()

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total

    model_1.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader_1:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_1(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/20 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/20 | Train Acc: 13.17% | Valid Acc: 16.18%
Epoch 2/20 | Train Acc: 18.82% | Valid Acc: 21.29%
Epoch 3/20 | Train Acc: 23.74% | Valid Acc: 27.65%
Epoch 4/20 | Train Acc: 27.66% | Valid Acc: 37.67%
Epoch 5/20 | Train Acc: 31.48% | Valid Acc: 29.29%
Epoch 6/20 | Train Acc: 33.02% | Valid Acc: 27.55%
Epoch 7/20 | Train Acc: 36.46% | Valid Acc: 23.41%
Epoch 8/20 | Train Acc: 39.70% | Valid Acc: 44.41%
Epoch 9/20 | Train Acc: 42.40% | Valid Acc: 39.60%
Epoch 10/20 | Train Acc: 45.78% | Valid Acc: 37.09%
Epoch 11/20 | Train Acc: 49.60% | Valid Acc: 31.70%
Epoch 12/20 | Train Acc: 54.00% | Valid Acc: 26.59%
Epoch 13/20 | Train Acc: 57.69% | Valid Acc: 51.25%
Epoch 14/20 | Train Acc: 61.26% | Valid Acc: 65.32%
Epoch 15/20 | Train Acc: 64.50% | Valid Acc: 70.04%
Epoch 16/20 | Train Acc: 68.71% | Valid Acc: 71.39%
Epoch 17/20 | Train Acc: 74.11% | Valid Acc: 76.78%
Epoch 18/20 | Train Acc: 77.42% | Valid Acc: 82.08%
Epoch 19/20 | Train Acc: 80.37% | Valid Acc: 82.95%
Epoch 20/20 | Train A

As we can see that increasing no. of epochs would help so lets do that only 

In [16]:
transform_2 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

dataset_2 = dset(image_paths=image_paths, transform=transform_2)

ts = int(0.75*len(dataset_2))
vs = len(dataset_2) - ts
generator = torch.Generator().manual_seed(42)
train_dataset, valid_dataset = random_split(dataset_2, [ts, vs], generator)

train_loader_2 = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

valid_loader_2 = DataLoader(
    valid_dataset,
    batch_size=16,
    shuffle=False
)

model_2 = torchvision.models.resnet34(weights="DEFAULT")

model_2 = add_stochastic_depth(model_2, max_drop_prob=0.2)

model_2.fc = nn.Linear(
    model_2.fc.in_features,
    num_classes
)

model_2 = model_2.to(device)

for param in model_2.parameters():
    param.requires_grad = True

train_aug_2 = K.AugmentationSequential(
    K.ColorJiggle(
        contrast=0.2,
        p=0.5
    ),
    K.RandomPlanckianJitter(
        mode="CIED",
        p=0.5
    )
).to(device)

targeted_aug_2 = K.AugmentationSequential(
    K.RandomRotation(
        degrees=10,
        p=0.5
    ),
    K.RandomAffine(
        degrees=0,
        scale=(0.9, 1.1),
        p=0.5
    ),
    K.RandomPerspective(
        distortion_scale=0.2,
        p=0.5
    ),
    K.ColorJiggle(
        brightness=0.2,
        contrast=0.2,
        p=0.5
    )
).to(device)

optimizer_2 = torch.optim.RMSprop(
    model_2.parameters(),
    lr=lr_head,
    weight_decay=1e-3
)

scheduler_2 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_2,
    max_lr=lr_head,
    epochs=40,
    steps_per_epoch=len(train_loader_2)
)

criterion = nn.CrossEntropyLoss()

for epoch in range(40):
    model_2.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_2:

        images = images.to(device)
        labels = labels.to(device)

        images = train_aug_2(images)

        mask = (labels == 35) | (labels == 36)

        if mask.any():
            images[mask] = targeted_aug_2(images[mask])

        optimizer_2.zero_grad()

        outputs = model_2(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer_2.step()

        scheduler_2.step()

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        train_correct += (predicted == labels).sum().item()

        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total

    model_2.eval()

    valid_loss = 0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():

        for images, labels in valid_loader_2:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_2(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = outputs.max(1)

            valid_correct += (predicted == labels).sum().item()

            valid_total += labels.size(0)

    valid_acc = 100 * valid_correct / valid_total

    print(
        f"Epoch {epoch+1}/40 | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Valid Acc: {valid_acc:.2f}%"
    )

Epoch 1/40 | Train Acc: 12.34% | Valid Acc: 23.22%
Epoch 2/40 | Train Acc: 22.20% | Valid Acc: 19.27%
Epoch 3/40 | Train Acc: 25.12% | Valid Acc: 18.69%
Epoch 4/40 | Train Acc: 30.04% | Valid Acc: 39.02%
Epoch 5/40 | Train Acc: 33.63% | Valid Acc: 26.30%
Epoch 6/40 | Train Acc: 38.00% | Valid Acc: 38.54%
Epoch 7/40 | Train Acc: 42.11% | Valid Acc: 30.54%
Epoch 8/40 | Train Acc: 46.06% | Valid Acc: 39.40%
Epoch 9/40 | Train Acc: 48.92% | Valid Acc: 47.88%
Epoch 10/40 | Train Acc: 50.95% | Valid Acc: 49.52%
Epoch 11/40 | Train Acc: 54.87% | Valid Acc: 44.41%
Epoch 12/40 | Train Acc: 59.81% | Valid Acc: 51.35%
Epoch 13/40 | Train Acc: 61.84% | Valid Acc: 49.13%
Epoch 14/40 | Train Acc: 63.70% | Valid Acc: 43.83%
Epoch 15/40 | Train Acc: 66.37% | Valid Acc: 38.92%
Epoch 16/40 | Train Acc: 70.03% | Valid Acc: 53.37%
Epoch 17/40 | Train Acc: 71.54% | Valid Acc: 38.25%
Epoch 18/40 | Train Acc: 73.31% | Valid Acc: 59.15%
Epoch 19/40 | Train Acc: 72.79% | Valid Acc: 69.65%
Epoch 20/40 | Train A

Still we can see that if we keep increasing the no. of epochs then it will benefit us but it is not beneficial as it is taking time and computation and then will give same results and so it is not that beneficial although this can train a more generalised model